# IO List validation tool

## Setup

In [2]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import List

import polars as pl
from openpyxl import load_workbook, Workbook
from openpyxl.worksheet.worksheet import Worksheet
from polars import DataFrame
from zero_data.io_list import IOResult


@dataclass
class IOValue:
    name: str
    data_type: str

    @staticmethod
    def from_json_path(json_path: str, data_type: str) -> "IOValue":
        return IOValue(
            name=json_path[2:],
            data_type=data_type,
        )


@dataclass
class IOTopic:
    topic: str
    fields: List[IOValue]


@dataclass
class IOResult:
    io_list: DataFrame
    topics: List[IOTopic]


_DATA_TYPES = {
    "Float": "REAL",
    "Bool": "BOOLEAN",
    "Uint32": "BIGINT",
    "Uint16": "INTEGER",
    # Typo in AMCS IO list R2.14
    "Unit16": "INTEGER",
    "Int32": "INTEGER",
    "Int16": "INTEGER",
    "String": "STRING",
}


class MarpowerReader:
    def __init__(self):
        self.topic_prefix = "marpower/"

    @staticmethod
    def _read_headers(workbook):
        """Determine the header starting from the first non-empty row of the Excel sheet"""
        last_main_header = None
        for col in workbook.columns:
            if col[0].value is None and col[1].value is None:
                break
            elif col[0].value is not None:
                last_main_header = col[0].value

            headers = [
                header
                for header in [last_main_header, col[1].value]
                if header is not None
            ]
            yield " ".join(headers)

    @staticmethod
    def _normalize_marpower_io_list(df: pl.DataFrame):
        """Normalize the IO list by renaming columns and filtering out unnecessary rows"""
        renamed_df = df.rename(lambda c: c.replace(" ", "_").lower())
        filter_df = (
            renamed_df.filter(pl.col("deleted").is_null())
            .filter(pl.col("system") != "SPARE")
            .filter(pl.col("tag") != "SPARE")
        )
        typed_df = (
            filter_df.with_columns(
                pl.col("target_type").replace_strict(_DATA_TYPES).alias("data_type")
            )
            .with_columns(tag=pl.col("tag").str.replace_all(r"-|\.", "_"))
        )
        return typed_df.select([
            "device",
            "tag",
            "yard_tag",
            "target_type",
            "terminal",
            "cabinet",
            "system",
            "description",
            "unit",
            "precision",
            "data_type",
            "mqtt_topic",
            "mqtt_json_path",
        ]).cast({
            "device": pl.String,
            "tag": pl.String,
            "yard_tag": pl.String,
            "target_type": pl.String,
            "terminal": pl.String,
            "cabinet": pl.String,
            "system": pl.String,
            "description": pl.String,
            "unit": pl.String,
            "precision": pl.String,
            "data_type": pl.String,
            "mqtt_topic": pl.String,
            "mqtt_json_path": pl.String
        })

    def _get_io_topics(self, df: pl.DataFrame) -> List[IOTopic]:
        """Get the IO topics from the DataFrame"""
        result = []
        for row in (
                df
                        .drop_nulls("mqtt_topic")
                        .group_by("mqtt_topic")
                        .agg(pl.col("mqtt_json_path"), pl.col("data_type"))
                        .iter_rows(named=True)
        ):
            topic = self.determine_topic(row)
            values = [
                IOValue.from_json_path(json_path=jp, data_type=dt)
                for jp, dt in zip(row["mqtt_json_path"], row["data_type"])
            ]
            result.append(IOTopic(topic, values))
        return result

    def determine_topic(self, row: dict) -> str:
        """Create the topic out of a fixed prefix and a name that is a function of the IO list"""
        return self.topic_prefix + row["mqtt_topic"].replace(" ", "-").replace("+", "").lower()

    @staticmethod
    def convert_value(val):
        if val is None:
            return None
        elif isinstance(val, str):
            return val
        elif int(val) == val:
            return str(int(val))
        else:
            return str(val)

    @classmethod
    def _read_bordered_column(cls, ws, col: int):
        """Read a column from the Excel sheet, returning only the values with borders"""
        last_val = None
        for cell in next(ws.iter_cols(col, col, 3)):
            if cell.border.top is not None and cell.border.top.style is not None:
                last_val = cls.convert_value(cell.value)
            yield last_val

    @classmethod
    def _read_marpower_excel(cls, path: Path) -> pl.DataFrame:
        """Read the AMCS Excel file and return a DataFrame"""
        workbook = load_workbook(path, data_only=True)
        headers = cls._read_headers(workbook["IO-List"])
        data = {
            header: cls._read_bordered_column(workbook["IO-List"], index + 1)
            for index, header in enumerate(headers)
        }
        return pl.DataFrame(data).filter(~pl.all_horizontal(pl.all().is_null()))

    def read_io_list(self, paths: List[Path]) -> IOResult:
        """Read the IO list from the given paths and return an IOResult"""
        io_lists = [
            self._normalize_marpower_io_list(self._read_marpower_excel(path)) for path in paths
        ]
        io_list = pl.concat(io_lists)

        topics = self._get_io_topics(io_list)

        return IOResult(io_list, topics)


# Read in IO Lists

In [46]:
reader = MarpowerReader()
io_list_files = [p for p in Path("../io_lists").glob("*.xlsx")]
io_lists = {p.name: reader.read_io_list([p]) for p in io_list_files}
for name, io_list in io_lists.items():
    print(f"Loaded IO list: {name} with {len(io_list.topics)} topics")

Loaded IO list: 52422003_3210_AMCS IO-List R2.14.xlsx with 622 topics


## Check tag duplicates

In [30]:
def check_tag_duplicates(io_result: IOResult):
    df = io_result.io_list
    tag_counts = df.group_by("tag").len("tag_count").sort("tag_count", descending=True)
    duplicates = tag_counts.filter(tag_counts["tag_count"] > 1)
    return duplicates


for name, io_list in io_lists.items():
    duplicates = check_tag_duplicates(io_list)
    print(f"{name} has {len(duplicates)} duplicate tags")


52422003_3210_AMCS IO-List R2.14.xlsx has 540 duplicate tags


## Schema For nested topics

In [33]:
def get_non_matching_nested_topics(io_result: IOResult, topic_prefix: str):
    power_tag_df = io_result.io_list.filter(pl.col("mqtt_topic").str.starts_with(topic_prefix))
    nr_of_fields = (
        power_tag_df.group_by(pl.col("mqtt_topic")).len("nr_of_fields")
        .group_by("nr_of_fields").len("len").sort("len", descending=True)
        .head(1).to_dict()["nr_of_fields"][0]
    )

    non_matching_topics = [t for t in io_result.topics if len(t.fields) != nr_of_fields]
    return non_matching_topics


for name, io_list in io_lists.items():
    print(f"IO List: {name}")
    non_matches_powertag = get_non_matching_nested_topics(io_list, "power-tag/")
    print(f"{name} has {len(non_matches_powertag)} non-matching power-tag topics")

    non_matches_firedetection = get_non_matching_nested_topics(io_list, "450000 FIREDETECTION/")
    print(f"{name} has {len(non_matches_firedetection)} non-matching fire detection topics")

IO List: 52422003_3210_AMCS IO-List R2.14.xlsx
52422003_3210_AMCS IO-List R2.14.xlsx has 429 non-matching power-tag topics
52422003_3210_AMCS IO-List R2.14.xlsx has 510 non-matching fire detection topics


## Internal validation

validate columns against relevant internal list and project definitions:
- System
- DataType

In [51]:
def headers(sheet: Worksheet) -> List[str]:
    return [c[0] for c in sheet.iter_cols(max_row=1) if c]


def iterate_excel_column(sheet: Worksheet, index: int, skip_extra_headers=0) -> List[str]:
    return [c.value for c in next(sheet.iter_cols(min_col=index + skip_extra_headers, max_col=index, min_row=2)) if
            c and c.value]


def get_definitions_from_workbook_tab(sheet: Worksheet) -> dict[str, List[str]]:
    _headers = headers(sheet)
    return {str(header.value): iterate_excel_column(sheet, header.col_idx) for header in _headers}


def get_definitions(workbook: Workbook) -> dict[str, List[str]]:
    project_definitions = get_definitions_from_workbook_tab(workbook["Project definitions"])
    internal_lists = get_definitions_from_workbook_tab(workbook["Internal lists"])
    return internal_lists | project_definitions


def validate_column(sheet: Worksheet, column_name: str, expected_values: List[str]) -> dict[str, int]:
    _headers = headers(sheet)
    col_idx = [h for h in _headers if h.value == column_name][0].col_idx
    column_values = iterate_excel_column(sheet, col_idx, skip_extra_headers=0)
    invalid_values = [v for v in column_values if v not in expected_values]
    return dict(Counter(invalid_values))



In [54]:
for io_list_path in io_list_files:

    workbook = load_workbook(io_list_path, data_only=True)
    definitions = get_definitions(workbook)
    print("System Column values not matching definitions:")
    print(validate_column(workbook["IO-List"], "System", definitions["Systems"]))
    print("Data Type Column values not matching definitions:")
    print(validate_column(workbook["IO-List"], "Target Type", definitions["Data Type"]))


System Column values not matching definitions:
{'450000 AMCS': 78, 'SPARE': 86, '210000 BILGE FIFI': 48, '090000 DOORS HATCHES': 58, '250000 FRESH WATER': 61, '340000 SEWAGE': 20, '750000 DECK EQUIP': 6, 'THERMODYNAMICA': 1, '220000 NOVEC': 4, '350000 VENTILATION': 53, '250000 TECHWATER': 23, 'HPU': 4, 'KVM SWITCHING': 18, '450000 24VDC SYSTEM': 22, '380000 SEAWATER': 9, '150000 PCS': 24, '450000 BURGLAR': 7, '290000 PNEUMATIC ': 1, '300000 DIRTY OIL': 1, '150000 PROPULSION': 4, '210000 GENERAL SERVICE': 3, '450000 NAVIGATION LIGHTS': 81, '170000 STEERING SYSTEM': 5, '450000 GAS DETECTION': 2, '450000 FIREDETECTION': 67, 'AC DISTR 10P0.1': 885, 'AC DISTR 10P0.3': 705, 'AC DISTR 10P1': 480, 'AC DISTR 10P2': 375, 'AC DISTR 10P3': 945, '450000 UNDERWLIGHTS': 288}
Data Type Column values not matching definitions:
{'Uint32': 28, 'Unit16': 61}
